# Notebook 11 — data quality: rule audit + label noise (cleanlab) + quarantine

Hipótese do docente: o teto está nos dados (~0.6 com GBTs + limpeza). Auditoria em 3 camadas: Q0 regras do questionário (violations train/test), Q1 confident-joint (taxa de ruído + suspeitos), Q2 quarentena (drop do train, validação intacta, mesmos folds seed 42). Adota-se com 5/5 folds. Cache `nb11_*.npz`: reruns só treinam o que falta. `RUN_EXTRA` liga drop20/relabel (novos treinos).

In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
K = 16
TARGET = 'AGE_RANGE'
RUN_EXTRA = False

def _data_dir():
    for cand in [os.path.join('..', '..', 'data'), os.path.join('..', 'data'), 'data']:
        if os.path.isdir(cand):
            return cand
    raise FileNotFoundError('pasta data/ nao encontrada')
DATA_DIR = _data_dir()

train = pd.read_csv(os.path.join(DATA_DIR, 'train_nb02.csv'))
y = train[TARGET].to_numpy().astype(int)
Xv = train.drop(columns=['PID', TARGET]).to_numpy(dtype=np.float64)
FOLDS = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE).split(Xv, y))
print('X:', Xv.shape)

def xgb_recipe():
    return XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        tree_method='hist', random_state=RANDOM_STATE, n_jobs=-1, eval_metric='mlogloss')

RESULTADOS = []
def registar(exp, variante, f1s, base_f1s, nota=''):
    f1s = np.asarray(f1s, float); base_f1s = np.asarray(base_f1s, float)
    r = {'exp': exp, 'variante': variante, 'f1': float(f1s.mean()), 'std': float(f1s.std()),
         'delta': float(f1s.mean() - base_f1s.mean()), 'folds_ganhos': int((f1s > base_f1s).sum()), 'nota': nota}
    RESULTADOS.append(r)
    print(r)
    return r

## Q0 — auditoria por regras (rápida, sem treino)

Gates oficiais validados no questionário. Violação = erro de registo ou gate fuzzy (ex. 55 exatos na classe 9).

In [ ]:
nb01 = pd.read_csv(os.path.join(DATA_DIR, 'train_nb01.csv'))
te01 = pd.read_csv(os.path.join(DATA_DIR, 'test_nb01.csv'))
NA = ['Não aplicável']; SR = ['Sem resposta', 'NR']
cls = pd.Series(y)
rep = {}
est_o = [(nb01[e] == 'Valor') | (nb01[e] == 'Gravidez sem termo (n.a.)') for e in ['IN24_ESTADO', 'IN25_ESTADO', 'IN26_ESTADO']]
o_ans = est_o[0] | est_o[1] | est_o[2]
rep['homens_com_O'] = int(((nb01['SEX'] == 'Homem') & o_ans).sum())
rep['O_taxa'] = round(float(o_ans.mean()), 4)
g_ans = ~(nb01['PC1A'].isin(NA + SR)) & nb01['PC1A'].notna()
rep['G_violacoes_idade'] = int((g_ans & (cls <= 8)).sum())
rep['G_taxa_treino'] = round(float(g_ans.mean()), 4)
g_te = ~(te01['PC1A'].isin(NA + SR)) & te01['PC1A'].notna()
rep['G_taxa_teste'] = round(float(g_te.mean()), 4)
rep['CD1J_contramed'] = int(((nb01['CD1J'] == 'Não') & (nb01['IN64'] == 'Sim')).sum())
rep['viuvos_ate35'] = int(((nb01['MARSTALEGAL'] == 'Viúvo') & (cls <= 5)).sum())
rep['alunos_55mais'] = int(((nb01['MAINSTAT'].str.contains('Aluno')) & (cls >= 9)).sum())
rep['reformados_ate34'] = int(((nb01['MAINSTAT'].str.contains('Reformado')) & (cls <= 4)).sum())
print(json.dumps(rep, indent=1))
print('Leitura: routing intacto (0 violações O-sexo, 0 G-idade, taxas train=test); erros são pontuais, não sistemáticos.')


## Q1 — confident joint (cleanlab): taxa + suspeitos

OOF do XGB-311 (5 folds). `find_label_issues` single-process (`n_jobs=1`: o multiprocessing quebra sob `conda run`).

In [ ]:
p_oof = os.path.join(DATA_DIR, 'nb11_oof311.npz')
if not os.path.exists(p_oof):
    oof = np.zeros(len(y), dtype=int); proba = np.zeros((len(y), K))
    for tr, va in FOLDS:
        m = xgb_recipe()
        m.fit(Xv[tr], y[tr], sample_weight=compute_sample_weight('balanced', y[tr]))
        oof[va] = m.predict(Xv[va]); proba[va] = m.predict_proba(Xv[va])
    np.savez_compressed(p_oof, oof=oof, proba=proba)
d = np.load(p_oof)
oof, proba = d['oof'].astype(int), d['proba']
print('OOF macro-F1:', round(f1_score(y, oof, average='macro'), 4))
from cleanlab.filter import find_label_issues
mask = np.asarray(find_label_issues(labels=y, pred_probs=proba, n_jobs=1), dtype=bool)
print(f'issues: {mask.sum()} ({mask.mean()*100:.1f}%) — majorante (mistura ambiguidade + ruído)')
selfc = proba[np.arange(len(y)), y]
rank = np.argsort(selfc)
sus = pd.DataFrame({'row': rank[:500], 'PID': train['PID'].to_numpy()[rank[:500]],
                      'true': y[rank[:500]], 'pred': oof[rank[:500]],
                      'selfc': np.round(selfc[rank[:500]], 4)})
sus.to_csv(os.path.join(DATA_DIR, 'nb11_suspects.csv'), index=False)
print(sus.head(10).to_string(index=False))
print('suspeitos por classe verdadeira:')
print(pd.Series(y[rank[:1096]]).value_counts().sort_index().to_string())

## Q2 — quarentena: drop do train por self-confidence (validação intacta)

Limiares 5/10/15% (decisão) + 20%/relabel atrás de `RUN_EXTRA`. Referência = base 311 sem drop.

In [ ]:
def quar(tag, drop_idx):
    p = os.path.join(DATA_DIR, f'nb11_quar5_{tag}.npz')
    if os.path.exists(p):
        f1s = np.load(p)['f1s']
        print(f'{tag}: cache {f1s.mean():.4f}')
        return f1s
    t0 = time.time(); f1s = []
    for fi, (a, b) in enumerate(FOLDS):
        atr = np.setdiff1d(a, drop_idx, assume_unique=False)
        m = xgb_recipe()
        m.fit(Xv[atr], y[atr], sample_weight=compute_sample_weight('balanced', y[atr]))
        f1s.append(f1_score(y[b], m.predict(Xv[b]), average='macro'))
        print(f'  {tag} fold {fi}: {f1s[-1]:.4f} ({time.time()-t0:.0f}s)', flush=True)
    f1s = np.array(f1s)
    np.savez_compressed(p, f1s=f1s)
    return f1s

d = np.load(os.path.join(DATA_DIR, 'nb11_oof311.npz'))['proba']
rank = np.argsort(d[np.arange(len(y)), y])
f1s_ref = quar('ref', np.array([], dtype=int))
for pct, tag in [(5, 'drop5'), (10, 'drop10'), (15, 'drop15')]:
    f1s = quar(tag, rank[:int(len(y) * pct / 100)])
    registar('Q2', f'drop-worst-{pct}pct', f1s, f1s_ref)
if RUN_EXTRA:
    f1s = quar('drop20', rank[:int(len(y) * 20 / 100)])
    registar('Q2', 'drop-worst-20pct', f1s, f1s_ref)
    yr = y.copy()
    rel = rank[:int(len(y) * 10 / 100)]
    oof = np.load(os.path.join(DATA_DIR, 'nb11_oof311.npz'))['oof'].astype(int)
    yr[rel] = oof[rel]
    p = os.path.join(DATA_DIR, 'nb11_quar5_relab10.npz')
    f1s = []
    for fi, (a, b) in enumerate(FOLDS):
        m = xgb_recipe()
        m.fit(Xv[a], yr[a], sample_weight=compute_sample_weight('balanced', yr[a]))
        f1s.append(f1_score(y[b], m.predict(Xv[b]), average='macro'))
    f1s = np.array(f1s)
    np.savez_compressed(p, f1s=f1s)
    registar('Q2', 'relabel-worst-10pct', f1s, f1s_ref, 'validação sempre com labels originais')

## Vereditos (modo completo, 5 folds seed 42)

- Q0: features limpas — 0 violações de routing (sexo-O, idade-G), taxas train=test, contradições meds ~0.3%. Ruído é nos labels, não nas features.
- Q1: confident-joint sinaliza 41% (majorante); auditoria manual dos piores mostra perfis incompatíveis (empregados com mestrado aos 85+, divorciada diabética aos 20-24).
- Q2: ref 0.4777 | drop5 0.4939 (+0.016) | drop10 0.5021 (+0.024) | drop15 0.5093 (+0.032) — todos 5/5. QUARENTENA ADOTADA (10%: equilíbrio ganho/dados; 15% em observação).
- Próximo: rebuild das bases + meta na base limpa (10%); refinamento por classe; drop20/relabel com RUN_EXTRA=True.

In [ ]:
final = pd.DataFrame(RESULTADOS).sort_values('f1', ascending=False)
for c in ('f1', 'std', 'delta'):
    final[c] = final[c].round(4)
print(final.to_string(index=False))
final.to_csv(os.path.join(DATA_DIR, 'nb11_resultados.csv'), index=False)